## ID3 (Iterative Dichotomiser 3) :

In [1]:
import math  # pour utiliser log2
from collections import Counter  # pour compter facilement les classes

# 1. Fonction Entropie
def entropy(data):
    labels = [row[-1] for row in data]  # récupérer la dernière colonne (les labels Oui/Non)
    total = len(labels)  # nombre total d'exemples
    counts = Counter(labels)  # compter combien de Oui et Non
    ent = 0  # initialiser l'entropie
    for count in counts.values():  # pour chaque classe (Oui, Non)
        p = count / total  # probabilité de la classe
        ent -= p * math.log2(p)  # formule de l'entropie
    return ent

# 2. Gain d'information
def information_gain(data, attr_index):
    total_entropy = entropy(data)  # entropie globale avant division
    total = len(data)  # nombre total d'exemples
    subsets = {}  # dictionnaire pour stocker les sous-ensembles
    for row in data:  # regrouper les lignes selon la valeur de l'attribut
        key = row[attr_index]  # valeur de l'attribut (ex: Soleil)
        subsets.setdefault(key, []).append(row)  # setdefault crée la liste si elle n'existe pas
        
    weighted_entropy = 0   # calcul de l'entropie pondérée

    # pour chaque sous-ensemble
    for subset in subsets.values():
        weighted_entropy += (len(subset) / total) * entropy(subset)  # proportion * entropie du sous-ensemble
        
    return total_entropy - weighted_entropy  # gain = entropy(data) - weighted_entropy (somme des proportions × entropies locales)

# 3. Choisir meilleur attribut
def best_attribute(data):
    n_attributes = len(data[0]) - 1   # nombre d'attributs (sans la colonne label)
    gains = [information_gain(data, i) for i in range(n_attributes)]   # calculer le gain pour chaque attribut
    return gains.index(max(gains))    # retourner l'indice de l'attribut avec gain max

# 4. Construire l’arbre
def build_tree(data, attributes):
    labels = [row[-1] for row in data]    # récupérer les labels

    # Cas 1 : toutes les classes identiques
    if labels.count(labels[0]) == len(labels):
        return labels[0]   # retourner Oui ou Non directement

    # Cas 2 : plus d'attributs disponibles
    if not attributes:
        return Counter(labels).most_common(1)[0][0]   # retourner la classe majoritaire

    # Choisir meilleur attribut
    best = best_attribute(data)   # indice du meilleur attribut
    best_attr_name = attributes[best]   # nom de l'attribut
    tree = {best_attr_name: {}}  # créer un arbre (dictionnaire)
    values = set(row[best] for row in data)  # récupérer toutes les valeurs possibles de cet attribut

    # pour chaque valeur (ex: Soleil, Pluie...)
    for value in values:
        subset = [row for row in data if row[best] == value]  # filtrer les lignes correspondant à cette valeur

        # si le sous-ensemble est vide
        if not subset:
            tree[best_attr_name][value] = Counter(labels).most_common(1)[0][0]  # mettre la classe majoritaire
        else:
            new_data = [row[:best] + row[best+1:] for row in subset]  # enlever l'attribut utilisé
            new_attributes = attributes[:best] + attributes[best+1:]  # enlever l'attribut de la liste
            tree[best_attr_name][value] = build_tree(new_data, new_attributes)  # appel récursif pour continuer l'arbre
            
    return tree   # retourner l'arbre construit

# 5. Exemple
data = [
    ["Soleil", "Chaud", "Haute", "Faible", "Non"],
    ["Soleil", "Chaud", "Haute", "Fort", "Non"],
    ["Nuageux", "Chaud",  "Haute", "Faible", "Oui"],
    ["Pluie", "Froid", "Normale", "Faible", "Oui"],
    ["Pluie", "Froid", "Normale", "Fort", "Non"],
    ["Nuageux", "Froid", "Normale", "Fort", "Oui"]
]

attributes = ["Temps", "Température", "Humidité", "Vent"]  # noms des attributs
tree = build_tree(data, attributes)  # construire l'arbre
print(tree) # afficher le résultat    

{'Temps': {'Soleil': 'Non', 'Pluie': {'Vent': {'Fort': 'Non', 'Faible': 'Oui'}}, 'Nuageux': 'Oui'}}


## C4.5 : 

In [5]:
# 1. Entropie
def entropy(data):
    labels = [row[-1] for row in data]  # récupérer la dernière colonne (les labels Oui/Non)
    total = len(labels)  # nombre total d'exemples
    counts = Counter(labels)  # compter Oui / Non
    ent = 0  # initialiser l'entropie
    for count in counts.values():  # pour chaque classe (Oui, Non)
        p = count / total  # probabilité de la classe
        ent -= p * math.log2(p)  # formule de l'entropie
    return ent  # retourner entropie

# 2. Split Info (spécifique C4.5)
def split_info(data, attr_index):
    total = len(data)   # nombre total
    counts = {}     # dictionnaire pour compter chaque valeur
    for row in data:    # nombre d’éléments pour chaque valeur de l’attribut (Sv)
        key = row[attr_index]
        counts[key] = counts.get(key, 0) + 1    # incrémenter
    si = 0     # initialisation
    for count in counts.values():
        p = count / total    # proportion de chaque groupe (Sv/S)
        si -= p * math.log2(p)    # formule SplitInfo (−∑v ∣Sv∣∣S∣ * log2 * ∣Sv∣∣S∣)
    return si

# 3. Gain Ratio
def gain_ratio(data, attr_index):
    total_entropy = entropy(data)  # entropie globale
    total = len(data)

    # regrouper les données selon l'attribut
    subsets = {}  
    for row in data:
        key = row[attr_index]
        subsets.setdefault(key, []).append(row)

    # calcul entropie après split
    weighted_entropy = 0
    for subset in subsets.values():
        weighted_entropy += (len(subset) / total) * entropy(subset)

    gain = total_entropy - weighted_entropy  # calcul gain

    # calcul SplitInfo
    si = split_info(data, attr_index)
    if si == 0:  # éviter division par zéro
        return 0
    return gain / si   # Gain Ratio final

# 4. Choisir meilleur attribut
def best_attribute(data):  
    n = len(data[0]) - 1    # nombre d'attributs (sans label)
    ratios = [gain_ratio(data, i) for i in range(n)]    # calcul GainRatio pour chaque attribut
    return ratios.index(max(ratios))    # retourner l'indice du meilleur

# 5. Construction de l'arbre
def build_tree(data, attributes):
    labels = [row[-1] for row in data]   # récupérer les classes

    # Cas 1 : données pures
    if labels.count(labels[0]) == len(labels):
        return labels[0]  # retourner Oui ou Non directement

    # Cas 2 : plus d'attributs
    if not attributes:
        return Counter(labels).most_common(1)[0][0]  # retourner classe majoritaire

    # Choisir meilleur attribut
    best = best_attribute(data)  # indice
    attr_name = attributes[best]  # nom
    tree = {attr_name: {}}  # créer nœud arbre
    values = set(row[best] for row in data)  # récupérer les valeurs possibles

    # pour chaque valeur de l’attribut
    for value in values:
        subset = [row for row in data if row[best] == value]  # créer sous-ensemble
        
        if not subset:  # si aucune donnée ne correspond à cette valeur, on retourne la classe la plus fréquente
            tree[attr_name][value] = Counter(labels).most_common(1)[0][0]  
        else:
            # enlever l'attribut utilisé
            new_data = [row[:best] + row[best+1:] for row in subset]
            new_attr = attributes[:best] + attributes[best+1:]
            tree[attr_name][value] = build_tree(new_data, new_attr)  # appel récursif

    return tree  # retourner arbre

## CART (Classification And Regression Trees) : 

In [4]:
# 1. Calcul Gini
def gini(data):
    labels = [row[-1] for row in data]    # récupérer la dernière colonne (classe)
    total = len(labels)    # nombre total d'exemples
    counts = Counter(labels)     # compter Oui / Non
    g = 1      # initialiser Gini à 1
    for c in counts.values():  
        p = c / total    # probabilité de la classe
        g -= p ** 2    # appliquer la formule Gini = 1 - somme(p²)

    return g  # retourner la valeur de Gini

# 2. Trouver meilleur split
def best_split(data):
    n_attr = len(data[0]) - 1   # nombre d'attributs (sans la classe)
    best_gini = float("inf")    # initialiser avec une grande valeur
    best_attr = None   # stocker le meilleur attribut

    # tester chaque attribut
    for i in range(n_attr):
        values = set(row[i] for row in data)   # valeurs possibles (ex: Soleil, Pluie)

        # CART teste des splits binaires : une valeur contre le reste
        for v in values:
            left = [row for row in data if row[i] == v]   # groupe gauche : lignes où attribut = v
            right = [row for row in data if row[i] != v]  # groupe droit : lignes où attribut != v

            # Gini pondéré
            g = (len(left)/len(data)) * gini(left) + (len(right)/len(data)) * gini(right)
            if g < best_gini:   # garder le meilleur split (Gini minimal)
                best_gini = g
                best_attr = (i, v)   # stocker (indice attribut, valeur)

    return best_attr  # retourner le meilleur split

# 3. Construire arbre
def build_tree(data):
    labels = [row[-1] for row in data]  # récupérer les classes

    # si pur → feuille
    if labels.count(labels[0]) == len(labels):
        return labels[0]   # retourner directement Oui ou Non
    attr, value = best_split(data)  # meilleur split

    # diviser les données
    left = [row for row in data if row[attr] == value]
    right = [row for row in data if row[attr] != value]

    # construire récursivement les sous-arbres
    return {
        f"{attr} == {value}": {  # condition du split
            "True": build_tree(left),  # branche gauche
            "False": build_tree(right) # branche droite
        }
    }